# Evaluation: Fine-tuned Mistral 7B vs Gemini 1.5 Flash vs Base Mistral 7B
Run this notebook on **Google Colab** with a T4 GPU (Runtime → Change runtime type → T4 GPU).

What this does:
1. Loads 100 test samples from your uploaded `test.jsonl`
2. Runs all 3 models on the same samples
3. Computes ROUGE-L and BERTScore
4. Saves `results.json` and prints the final comparison table

In [1]:
!pip install -q \
    transformers==4.46.0 \
    peft==0.11.1 \
    bitsandbytes==0.45.5 \
    accelerate==0.34.0 \
    sentencepiece==0.2.0 \
    evaluate==0.4.3 \
    rouge-score==0.1.2 \
    bert-score==0.3.13 \
    datasets==2.19.0 \
    google-generativeai

print('Done.')

Done.


In [2]:
#Upload test.jsonl
from google.colab import files
print('Select your test.jsonl file...')
uploaded = files.upload()   # a file picker will appear
print('Uploaded:', list(uploaded.keys()))

Select your test.jsonl file...


Saving test.jsonl to test (1).jsonl
Uploaded: ['test (1).jsonl']


In [17]:
# Config
# GEMINI_KEY      = ''
FINETUNED_MODEL = 'kk014/mistral-7b-docstring'
BASE_MODEL      = 'mistralai/Mistral-7B-v0.1'
TEST_FILE       = 'test.jsonl'
EVAL_SAMPLES    = 100
MAX_NEW_TOKENS  = 150

PROMPT_TEMPLATE = (
    'You are a Python documentation expert. '
    'Write a clear, concise NumPy-style docstring for the following Python function.\n\n'
    '### Function:\n{function_code}\n\n'
    '### Docstring:'
)
print('Config set.')

Config set.


In [4]:
#Imports
import os, json, time, torch
from tqdm import tqdm
import evaluate as hf_evaluate
from bert_score import score as bert_score_fn
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')

CUDA available : True
GPU            : Tesla T4


In [5]:
# Load test samples
samples = []
with open(TEST_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        samples.append(json.loads(line))
        if len(samples) >= EVAL_SAMPLES:
            break

references = [s['ground_truth_docstring'] for s in samples]
print(f'Loaded {len(samples)} test samples.')
print(f'\nExample function:')
print(samples[0]['function_code'][:200])
print(f'\nGround truth docstring:')
print(references[0][:200])

Loaded 100 test samples.

Example function:
def python_logging_error_handler(level, root_says_abort, location, msg):
    from ..utils import quickroot as QROOT
    if not Initialized.value:
    try:
        QROOT.kTRUE
    except AttributeError

Ground truth docstring:
A python error handler for ROOT which maps ROOT's errors and warnings on
    to python's.


In [6]:
# Helper: load HuggingFace model in 4-bit
def load_hf_model(model_name):
    print(f'\nLoading: {model_name}')
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = 'right'

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
    if model_name != BASE_MODEL:
        print(f'Attaching LoRA adapter: {model_name}')
        model = PeftModel.from_pretrained(base, model_name)
    else:
        model = base

    model.eval()
    print('Ready.')
    return model, tokenizer


# Helper: generate docstring from HF model
def generate_hf(model, tokenizer, function_code):
    prompt = PROMPT_TEMPLATE.format(function_code=function_code)
    inputs = tokenizer(
        prompt, return_tensors='pt',
        truncation=True, max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    docstring  = generated[len(prompt):].strip()
    # Cut off repetition after first blank line
    lines, clean = docstring.split('\n'), []
    for line in lines:
        clean.append(line)
        if len(clean) > 3 and line.strip() == '':
            break
    return '\n'.join(clean).strip()


# Helper: generate docstring from Gemini
def generate_gemini(model_client, function_code):
    prompt = PROMPT_TEMPLATE.format(function_code=function_code)
    try:
        response = model_client.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        print(f'  Gemini error: {e}')
        time.sleep(5)
        return ''


# Helper: compute ROUGE-L + BERTScore
def compute_metrics(predictions, references):
    valid = [(p, r) for p, r in zip(predictions, references) if p.strip()]
    preds, refs = zip(*valid)
    preds, refs = list(preds), list(refs)

    rouge       = hf_evaluate.load('rouge')
    rouge_l     = rouge.compute(predictions=preds, references=refs)['rougeL']

    print('  Computing BERTScore...')
    _, _, F1    = bert_score_fn(preds, refs, lang='en',
                                model_type='distilbert-base-uncased',
                                verbose=False)
    bertscore   = F1.mean().item()

    return {'rouge_l': round(rouge_l, 4), 'bertscore_f1': round(bertscore, 4)}


print('Helpers defined.')

Helpers defined.


In [7]:
# Run fine-tuned Mistral 7B
ft_model, ft_tokenizer = load_hf_model(FINETUNED_MODEL)

ft_preds = []
for s in tqdm(samples, desc='Fine-tuned Mistral'):
    ft_preds.append(generate_hf(ft_model, ft_tokenizer, s['function_code']))

print('Computing metrics...')
ft_metrics = compute_metrics(ft_preds, references)
print(f'Fine-tuned Mistral 7B → ROUGE-L: {ft_metrics["rouge_l"]}  BERTScore: {ft_metrics["bertscore_f1"]}')

# Free GPU memory
del ft_model
torch.cuda.empty_cache()


Loading: kk014/mistral-7b-docstring


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Attaching LoRA adapter: kk014/mistral-7b-docstring
Ready.


Fine-tuned Mistral: 100%|██████████| 100/100 [33:29<00:00, 20.09s/it]


...

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Fine-tuned Mistral 7B → ROUGE-L: 0.2033  BERTScore: 0.7739


In [22]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.7 MB/s eta 0:00:00


In [24]:
# Cell 8 — Run Llama 3.1 70B via Groq
from google.colab import userdata
from groq import Groq
import time


GROQ_KEY    = userdata.get('GROQ_KEY')   # add to Colab Secrets as GROQ_KEY
groq_client = Groq(api_key=GROQ_KEY)

def generate_groq(client, function_code):
    prompt = PROMPT_TEMPLATE.format(function_code=function_code)
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150,
            temperature=0.1,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"  Groq error: {e}")
        time.sleep(5)
        return ""

groq_preds = []
for s in tqdm(samples, desc="Llama 3.1 70B (Groq)"):
    groq_preds.append(generate_groq(groq_client, s["function_code"]))
    time.sleep(0.5)   # free tier: 30 req/min

print("Computing metrics...")
groq_metrics = compute_metrics(groq_preds, references)
print(f"Llama 3.1 70B → ROUGE-L: {groq_metrics['rouge_l']}  BERTScore: {groq_metrics['bertscore_f1']}")

Llama 3.1 70B (Groq): 100%|██████████| 100/100 [04:26<00:00,  2.66s/it]


Computing metrics...
  Computing BERTScore...
Llama 3.1 70B → ROUGE-L: 0.1715  BERTScore: 0.7594


In [25]:
# Run base Mistral 7B (no fine-tuning)
base_model, base_tokenizer = load_hf_model(BASE_MODEL)

base_preds = []
for s in tqdm(samples, desc='Base Mistral 7B'):
    base_preds.append(generate_hf(base_model, base_tokenizer, s['function_code']))

print('Computing metrics...')
base_metrics = compute_metrics(base_preds, references)
print(f'Base Mistral 7B → ROUGE-L: {base_metrics["rouge_l"]}  BERTScore: {base_metrics["bertscore_f1"]}')

del base_model
torch.cuda.empty_cache()


Loading: mistralai/Mistral-7B-v0.1


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Ready.


Base Mistral 7B: 100%|██████████| 100/100 [18:41<00:00, 11.22s/it]


Computing metrics...
  Computing BERTScore...
Base Mistral 7B → ROUGE-L: 0.1102  BERTScore: 0.7118


In [26]:
# Cell 10 — Final results table
all_results = {
    'Mistral 7B (fine-tuned)':        ft_metrics,
    'Llama 3.3 70B via Groq':          groq_metrics,
    'Mistral 7B (base, no fine-tune)': base_metrics,
}

print('\n' + '=' * 62)
print(f'{"Model":<35} {"ROUGE-L":>8} {"BERTScore":>10}')
print('-' * 62)
for name, m in all_results.items():
    print(f'{name:<35} {m["rouge_l"]:>8.4f} {m["bertscore_f1"]:>10.4f}')
print('=' * 62)

# Verdict
if ft_metrics['rouge_l'] > groq_metrics['rouge_l']:
    print(f'\n✓ Fine-tuned 7B BEATS Llama 3.3 70B on ROUGE-L!')
    print(f'  ({ft_metrics["rouge_l"]} vs {groq_metrics["rouge_l"]})')
else:
    print(f'\n✗ Llama 3.3 70B wins on ROUGE-L')

if ft_metrics['bertscore_f1'] > groq_metrics['bertscore_f1']:
    print(f'✓ Fine-tuned 7B BEATS Llama 3.3 70B on BERTScore!')
    print(f'  ({ft_metrics["bertscore_f1"]} vs {groq_metrics["bertscore_f1"]})')


Model                                ROUGE-L  BERTScore
--------------------------------------------------------------
Mistral 7B (fine-tuned)               0.2033     0.7739
Llama 3.3 70B via Groq                0.1715     0.7594
Mistral 7B (base, no fine-tune)       0.1102     0.7118

✓ Fine-tuned 7B BEATS Llama 3.3 70B on ROUGE-L!
  (0.2033 vs 0.1715)
✓ Fine-tuned 7B BEATS Llama 3.3 70B on BERTScore!
  (0.7739 vs 0.7594)


In [27]:
# Save results + markdown table
import json

results_data = {
    'results': all_results,
    'predictions': {
        'finetuned': ft_preds,
        'groq':      groq_preds,
        'base':      base_preds,
    },
    'references': references,
}
with open('results.json', 'w') as f:
    json.dump(results_data, f, indent=2)

md = [
    '## Evaluation Results\n\n',
    f'Evaluated on {EVAL_SAMPLES} held-out Python functions from CodeSearchNet.\n\n',
    '| Model | ROUGE-L | BERTScore F1 |\n',
    '|---|---|---|\n',
]
for name, m in all_results.items():
    md.append(f'| {name} | {m["rouge_l"]:.4f} | {m["bertscore_f1"]:.4f} |\n')

with open('summary.txt', 'w') as f:
    f.writelines(md)

print('Saved results.json and summary.txt')

from google.colab import files
files.download('results.json')
files.download('summary.txt')
print('Downloaded! Save both to mistral-docstring/eval/')

Saved results.json and summary.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded! Save both to mistral-docstring/eval/
